In [ ]:
# --timeframe 1d   : Таймфрейм свечей (дневные данные).
# --start-year 2000: Глубина загрузки истории (начиная с 2000 года).
# --workers 6      : Количество параллельных потоков для ускорения загрузки.

!python -m _tools.update_market_data --timeframe 1d --start-year 2000 --workers 6
!python -m _tools.update_macro --timeframe 1d --start-year 2000 --workers 6
# Выполняет комплексную проверку целостности, отсутствия пропусков и корректности OHLCV данных.
!python -m _tools.check_data_quality

In [ ]:
# --timeframe 1d       : Интервал данных — дневные свечи.
# --lookback 60        : Глубина истории — модель смотрит на 60 дней назад.
# --horizon 10         : Горизонт прогноза — ищем выход по барьерам в течение 10 дней.
# --auto               : Режим автоматического расчета уровней TP/SL на основе волатильности.
# --percentile 75      : Перцентиль волатильности для отсечения аномальных выбросов при авто-разметке.
# --init_split         : Дата начала первого разделения данных на Train и Val.
# --val_interval 2     : Продолжительность валидационного периода в годах.
# --split_interval 2   : Шаг смещения окна Walk-Forward в годах.
# --endpoint           : Дата окончания формирования всех временных интервалов.
# --corr_threshold     : Порог удаления коррелирующих признаков (убираем дубликаты > 85%).
# --cum_threshold      : Порог кумулятивной важности (оставляем топ фичей, дающих 99% влияния).
# --force              : Раскомментируйте параметр ниже для полной перезаписи кэшированных данных.

!python -m _tools.init_dataset \
    --timeframe 1d \
    --lookback 60 \
    --horizon 10 \
    --auto \
    --percentile 75 \
    --init_split 2010-01-01 \
    --val_interval 2 \
    --split_interval 2 \
    --endpoint 2024-01-01 \
    --corr_threshold 0.85 \
    --cum_threshold 0.99 \
    #--force

In [24]:
!python -m _tools.verify_data

I0000 00:00:1776764278.333671  261702 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1776764279.941755  261702 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
✅ Расширенный аудит завершен: /home/restorator/trader_test/data_audit_report.txt


In [ ]:
#!rm -rf data/processed

In [ ]:
import os
import subprocess
from pathlib import Path

DATASET_DIR = Path("data/processed/2000_2026_1d")
print("🚀 Запуск массового обучения моделей (Walk-Forward)...")

# Получаем список папок fold_
folds = sorted([d for d in DATASET_DIR.glob("fold_*") if d.is_dir()])

try:
    for fold_dir in folds:
        fold_name = fold_dir.name
        models_dir = fold_dir / "models"
        
        # Проверка на существование готовой модели
        if list(models_dir.glob("*.keras")):
            print(f"⏭️ [{fold_name}] Модель уже обучена. Пропускаем...")
            continue
            
        print("="*60)
        print(f"🔥 Обучение нейросети для: {fold_name}")
        print("="*60)
        
        # Формируем команду вызова
        # ВНИМАНИЕ: я снизил batch_size до 4096, т.к. 8192 часто вызывает 
        # падение скорости или ошибки памяти на длинных последовательностях
        cmd = [
            "python", "-m", "_tools.train_model",
            "--dataset_dir", str(DATASET_DIR),
            "--fold", fold_name,
            "--runs", "50",
            "--batch_size", "8192", 
            "--epochs", "50",
            "--l2_reg", "1e-4",
            "--lr", "1e-3"
        ]
        
        # Запускаем процесс и позволяем ему выводить логи в реальном времени
        process = subprocess.Popen(cmd)
        
        # Ждем завершения, но позволяем Jupyter перехватить прерывание
        process.wait()
        
        print(f"✅ [{fold_name}] Завершен!")

except KeyboardInterrupt:
    print("\n🛑 Остановка пайплайна пользователем!")
    if 'process' in locals():
        process.terminate() # Мягкая остановка текущего процесса
        print("⏳ Завершаем текущий фолд...")
except Exception as e:
    print(f"❌ Ошибка: {e}")

print("🎉 Процесс полностью остановлен.")

🚀 Запуск массового обучения моделей (Walk-Forward)...
⏭️ [fold_2010] Модель уже обучена. Пропускаем...
⏭️ [fold_2012] Модель уже обучена. Пропускаем...
🔥 Обучение нейросети для: fold_2014
✅ Mixed precision включена!
✅ Динамическое выделение видеопамяти включено!
🚀 Старт обучения. Фолд: [fold_2014]
📊 Форма данных: [Lookback: 60, Features: 71]
⚙️ Расчет идеальных весов классов...
   Баланс: SL(0)=17372, Hold(1)=42222, TP(2)=17088
   Веса:   SL(0)=1.47, Hold(1)=0.61, TP(2)=1.50
⏳ Подготовка конвейера данных...

--------------------------------------------------
🔄 ИТЕРАЦИЯ 1/50 (Рекорд фолда: 0.00%)
--------------------------------------------------
Epoch 1/50
10/10 - 8s - 799ms/step - accuracy: 0.4345 - loss: 1.1632 - val_accuracy: 0.4979 - val_loss: 1.1239
Epoch 2/50
10/10 - 2s - 206ms/step - accuracy: 0.5140 - loss: 1.0603 - val_accuracy: 0.4155 - val_loss: 1.1627
Epoch 3/50
10/10 - 2s - 201ms/step - accuracy: 0.5178 - loss: 1.0340 - val_accuracy: 0.4782 - val_loss: 1.1566
Epoch 4/50
10

In [ ]:
!python -m _tools.prepare_rl_env

In [ ]:
!python -m _tools.train_rllib_pbt --population 4

In [ ]:
!python -m _tools.evaluate